<a href="https://colab.research.google.com/github/rohinerd/document-qa-rag/blob/main/document_qa_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("hello")

hello


In [3]:
!pip install sentence-transformers chromadb mlflow -q

In [19]:
!wget https://github.com/rohinerd/document-qa-rag/blob/main/data/flexible-working.txt

--2026-11-01 00:47:40--  https://github.com/rohinerd/document-qa-rag/blob/main/data/flexible-working.txt
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘flexible-working.txt.1’

flexible-working.tx     [ <=>                ] 235.37K  --.-KB/s    in 0.03s   

2026-11-01 00:47:41 (7.16 MB/s) - ‘flexible-working.txt.1’ saved [241018]



In [18]:
!wget -q https://raw.githubusercontent.com/rohinerd/document-qa-rag/refs/heads/main/data/flexible-working.txt -O flexible-working.txt

In [22]:
with open("flexible-working.txt", encoding="utf-8") as f:
    text = f.read()

print(len(text), "characters")
print(text[:250])

7109 characters
Flexible working

1. Overview
Flexible working is a way of working that suits an employee’s needs, for example having flexible start and finish times, or working from home.

Flexible working rules are different in Northern Ireland.

All employees hav


In [30]:
def chunk_text(text, size=350, overlap=50):
    words = text.split()
    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(text)
print(len(chunks), "chunks")
print(chunks[0][:350])

4 chunks
Flexible working 1. Overview Flexible working is a way of working that suits an employee’s needs, for example having flexible start and finish times, or working from home. Flexible working rules are different in Northern Ireland. All employees have the legal right to request flexible working. Employees can request a change to: the number of hours t


In [31]:
print(len(text.split()), "words")
print(text[-300:])

1145 words
ng their request was treated as withdrawn
the date the employer should have responded to their request (but failed to do so)
If the problem happened before 1 October 2026, they may need to complain within 3 months.

If an employer or employee is unsure of their rights, they should get legal advice.



In [32]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.create_collection("flexible_working")

embeddings = model.encode(chunks).tolist()
collection.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
)

def retrieve(question, k=3):
    query_embedding = model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return results["documents"][0]

for q in [
    "Who has the legal right to request flexible working?",
    "From when can an employee make a request?",
    "Which aspects of work can an employee ask to change?",
]:
    print("Q:", q)
    for passage in retrieve(q):
        print("  -", passage[:120], "...")
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q: Who has the legal right to request flexible working?
  - Flexible working 1. Overview Flexible working is a way of working that suits an employee’s needs, for example having fle ...
  - limits) but works certain ‘core hours’, for example 10am to 4pm every day. Annualised hours The employee has to work a c ...
  - request a statement that this is a statutory request for flexible working details of how the employee wants to work flex ...

Q: From when can an employee make a request?
  - request a statement that this is a statutory request for flexible working details of how the employee wants to work flex ...
  - demand there’s a lack of work to do during the proposed working times the business is planning changes to the workforce  ...
  - Flexible working 1. Overview Flexible working is a way of working that suits an employee’s needs, for example having fle ...

Q: Which aspects of work can an employee ask to change?
  - limits) but works certain ‘core hours’, for example 10am to 4pm

In [35]:
def chunk_text(t, size=120, overlap=20):
    words = t.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step) if words[i:i + size]]

headings = ["Types of flexible working", "Applying for flexible working", "After the application", "Appeals"]

body_start = text.index("Flexible working is a way of working")
starts = [("Overview", body_start)]
pos = body_start
for name in headings:
    idx = text.index(name, pos)
    starts.append((name, idx))
    pos = idx + len(name)

bounds = starts + [("END", len(text))]
sections = {name: text[s:e] for (name, s), (_, e) in zip(bounds, bounds[1:])}

chunks, labels = [], []
for name, body in sections.items():
    for c in chunk_text(body):
        chunks.append(c)
        labels.append(name)

for name, body in sections.items():
    print(f"{name}: {len(body.split())} words")
print(len(chunks), "chunks")

Overview: 183 words
Types of flexible working: 223 words
Applying for flexible working: 293 words
After the application: 231 words
Appeals: 211 words
14 chunks


In [36]:
collection2 = client.create_collection("flexible_working_sections")
collection2.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=model.encode(chunks).tolist(),
    documents=chunks,
    metadatas=[{"section": l} for l in labels],
)

def retrieve(question, k=3):
    res = collection2.query(query_embeddings=model.encode([question]).tolist(), n_results=k)
    return [(m["section"], d) for m, d in zip(res["metadatas"][0], res["documents"][0])]

In [38]:
import csv, io, urllib.request

EVAL_URL = "https://raw.githubusercontent.com/rohinerd/document-qa-rag/main/data/eval.csv"
rows = list(csv.DictReader(io.StringIO(urllib.request.urlopen(EVAL_URL).read().decode("utf-8"))))

hits1 = hits3 = 0
for row in rows:
    found = [s for s, _ in retrieve(row["question"], k=3)]
    hit1 = found[0] == row["source_section"]
    hit3 = row["source_section"] in found
    hits1 += hit1
    hits3 += hit3
    print(f'{row["id"]:>2}  top1={hit1}  top3={hit3}  expected={row["source_section"]}  got={found}')

print(f"\nTop-1 section hit rate: {hits1}/{len(rows)}")
print(f"Top-3 section hit rate: {hits3}/{len(rows)}")

 1  top1=True  top3=True  expected=Overview  got=['Overview', 'Applying for flexible working', 'Applying for flexible working']
 2  top1=False  top3=True  expected=Overview  got=['After the application', 'Overview', 'Appeals']
 3  top1=False  top3=False  expected=Overview  got=['Applying for flexible working', 'After the application', 'Applying for flexible working']
 4  top1=True  top3=True  expected=Overview  got=['Overview', 'Applying for flexible working', 'After the application']
 5  top1=True  top3=True  expected=Overview  got=['Overview', 'Overview', 'Applying for flexible working']
 6  top1=False  top3=False  expected=Overview  got=['Appeals', 'Appeals', 'Applying for flexible working']
 7  top1=True  top3=True  expected=Overview  got=['Overview', 'Appeals', 'After the application']
 8  top1=True  top3=True  expected=Overview  got=['Overview', 'Overview', 'After the application']
 9  top1=True  top3=True  expected=Appeals  got=['Appeals', 'After the application', 'After the app